<a href="https://www.kaggle.com/code/ahmedenayet/model-development-updated?scriptVersionId=373691295" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [2]:

!pip install -q gdown

import gdown
import os

# Folder to save the downloaded files
output_dir = "/kaggle/working/prepared_data"
os.makedirs(output_dir, exist_ok=True)

# Google Drive file IDs
files = {
    "validation.npy": "1C4oC5MtjUUrWj0WGH0iagv5A8nkKXkZF",
    "validation_labels.npy": "12aNe7CAD2BcuR6SMw67fgW5YbHUtFuB7",
    "train.npy": "1w1GxSpmeEebsTIDWlGiS_HXC23ZpJLOK",
    "train_labels.npy": "1YBjDvRMNJzlt7QML6VYINLTGMQnpNgsU",
    "test.npy": "1BgVcsn5A29CKwqi4suPC5dNyfZoYnTz9",
    "test_labels.npy": "1LCrW9VOk9UuyyMAr2C4MOSjMNbX6u__U"
}

# Download all files
for filename, file_id in files.items():
    output_path = os.path.join(output_dir, filename)

    print(f"\nDownloading {filename}...")

    result = gdown.download(
        id=file_id,
        output=output_path,
        quiet=False
    )

    if result is None or not os.path.exists(output_path):
        print(f"FAILED: {filename}")
    else:
        print(f"SUCCESS: {filename}")

print("\nDownload process finished!")


Downloading...
From (original): https://drive.google.com/uc?id=1C4oC5MtjUUrWj0WGH0iagv5A8nkKXkZF
From (redirected): https://drive.google.com/uc?id=1C4oC5MtjUUrWj0WGH0iagv5A8nkKXkZF&confirm=t&uuid=e412148a-b39d-41ce-89ef-d65525fde421
To: /kaggle/working/prepared_data/validation.npy
100%|██████████| 174M/174M [00:02<00:00, 60.5MB/s] 


SUCCESS: validation.npy



Downloading...
From: https://drive.google.com/uc?id=12aNe7CAD2BcuR6SMw67fgW5YbHUtFuB7
To: /kaggle/working/prepared_data/validation_labels.npy
100%|██████████| 28.4k/28.4k [00:00<00:00, 33.1MB/s]


SUCCESS: validation_labels.npy



Downloading...
From (original): https://drive.google.com/uc?id=1w1GxSpmeEebsTIDWlGiS_HXC23ZpJLOK
From (redirected): https://drive.google.com/uc?id=1w1GxSpmeEebsTIDWlGiS_HXC23ZpJLOK&confirm=t&uuid=9f12b837-5ace-4d23-9256-a1aed3e7a69e
To: /kaggle/working/prepared_data/train.npy
100%|██████████| 810M/810M [00:04<00:00, 164MB/s]  


SUCCESS: train.npy



... (2 outputs left out)

Downloading...
From (original): https://drive.google.com/uc?id=1BgVcsn5A29CKwqi4suPC5dNyfZoYnTz9
From (redirected): https://drive.google.com/uc?id=1BgVcsn5A29CKwqi4suPC5dNyfZoYnTz9&confirm=t&uuid=baf87f8f-b68e-4d56-86d9-1dfbd349f179
To: /kaggle/working/prepared_data/test.npy
100%|██████████| 174M/174M [00:01<00:00, 104MB/s]  


SUCCESS: test.npy



Downloading...
From: https://drive.google.com/uc?id=1LCrW9VOk9UuyyMAr2C4MOSjMNbX6u__U
To: /kaggle/working/prepared_data/test_labels.npy
100%|██████████| 28.4k/28.4k [00:00<00:00, 32.3MB/s]

SUCCESS: test_labels.npy

Download process finished!


In [4]:

import os

for filename in sorted(os.listdir(output_dir)):
    path = os.path.join(output_dir, filename)
    size_mb = os.path.getsize(path) / (1024 ** 2)
    print(f"{filename}: {size_mb:.2f} MB")


test.npy: 165.66 MB
test_labels.npy: 0.03 MB
train.npy: 772.83 MB
train_labels.npy: 0.13 MB
validation.npy: 165.61 MB
validation_labels.npy: 0.03 MB


In [5]:

import numpy as np

X_train = np.load(os.path.join(output_dir, "train.npy"), mmap_mode="r")
y_train = np.load(os.path.join(output_dir, "train_labels.npy"))

X_val = np.load(os.path.join(output_dir, "validation.npy"), mmap_mode="r")
y_val = np.load(os.path.join(output_dir, "validation_labels.npy"))

X_test = np.load(os.path.join(output_dir, "test.npy"), mmap_mode="r")
y_test = np.load(os.path.join(output_dir, "test_labels.npy"))

print("Train:", X_train.shape, y_train.shape)
print("Validation:", X_val.shape, y_val.shape)
print("Test:", X_test.shape, y_test.shape)


Train: (16487, 128, 128, 3) (16487,)
Validation: (3533, 128, 128, 3) (3533,)
Test: (3534, 128, 128, 3) (3534,)


In [1]:

import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.optimizers import Adam

print("TensorFlow version:", tf.__version__)
print("GPU available:", tf.config.list_physical_devices("GPU"))


TensorFlow version: 2.21.0
GPU available: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU'), PhysicalDevice(name='/physical_device:GPU:1', device_type='GPU')]


In [2]:
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models


def build_cnn(input_shape=(128, 128, 3), num_classes=10):
    """Build a baseline CNN for 10-class animal image classification."""
    inputs = layers.Input(shape=input_shape)

    # Block 1: 32 filters
    x = layers.Conv2D(32, (3, 3), padding="same", activation="relu")(inputs)
    x = layers.MaxPooling2D((2, 2))(x)

    # Block 2: 64 filters
    x = layers.Conv2D(64, (3, 3), padding="same", activation="relu")(x)
    x = layers.MaxPooling2D((2, 2))(x)

    # Block 3: 128 filters
    x = layers.Conv2D(128, (3, 3), padding="same", activation="relu")(x)
    x = layers.MaxPooling2D((2, 2))(x)

    # Head
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(64, activation="relu")(x)
    outputs = layers.Dense(num_classes, activation="softmax")(x)

    model = models.Model(inputs=inputs, outputs=outputs, name="baseline_cnn")
    return model


if __name__ == "__main__":
    model = build_cnn(input_shape=(128, 128, 3), num_classes=10)
    model.summary()

    # Pass a single dummy RGB image (batch size = 1) to verify output shape and probabilities
    sample_image = np.random.rand(1, 128, 128, 3).astype(np.float32)
    predictions = model(sample_image, training=False).numpy()

    print("Sample output shape:", predictions.shape)  # Expected: (1, 10)
    print("Probabilities sum:", float(np.sum(predictions[0])))  # Expected: ~1.0
    print("Sample probabilities:", predictions[0])

Model: "baseline_cnn"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 128, 128, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d (Conv2D)                 │ (None, 128, 128, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 64, 64, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 64, 64, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 32, 32, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 32, 32, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 16, 16, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 128)            │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 10)             │           650 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 102,154 (399.04 KB)

 Trainable params: 102,154 (399.04 KB)

 Non-trainable params: 0 (0.00 B)

Sample output shape: (1, 10)
Probabilities sum: 1.0
Sample probabilities: [0.10974894 0.0991309  0.10851722 0.09400905 0.09509332 0.10247866
 0.09754948 0.10234079 0.08915058 0.1019811 ]


In [3]:

model = build_cnn(
    input_shape=(128, 128, 3),
    num_classes=10
)

model.summary()

# Test with a dummy image
sample_image = np.random.rand(
    1, 128, 128, 3
).astype(np.float32)

predictions = model(sample_image, training=False).numpy()

print("Output shape:", predictions.shape)
print("Probabilities sum:", np.sum(predictions[0]))
print("Probabilities:", predictions[0])


Model: "baseline_cnn"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, 128, 128, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 128, 128, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_3 (MaxPooling2D)  │ (None, 64, 64, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_4 (Conv2D)               │ (None, 64, 64, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_4 (MaxPooling2D)  │ (None, 32, 32, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_5 (Conv2D)               │ (None, 32, 32, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_5 (MaxPooling2D)  │ (None, 16, 16, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_1      │ (None, 128)            │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 10)             │           650 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 102,154 (399.04 KB)

 Trainable params: 102,154 (399.04 KB)

 Non-trainable params: 0 (0.00 B)

Output shape: (1, 10)
Probabilities sum: 1.0000001
Probabilities: [0.08920316 0.10416558 0.09532193 0.10903595 0.10253623 0.09292577
 0.09934301 0.09265857 0.09510354 0.11970629]


In [7]:

model = compile_model(model)


Model compiled successfully!
Optimizer: Adam
Loss: Sparse Categorical Crossentropy
Metrics: Accuracy


In [6]:

import tensorflow as tf
from tensorflow.keras.optimizers import Adam

def compile_model(model):
    model.compile(
        optimizer=Adam(),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )

    print("Model compiled successfully!")
    print("Optimizer: Adam")
    print("Loss: Sparse Categorical Crossentropy")
    print("Metrics: Accuracy")

    return model
